In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.physicians_raw LIMIT 10;

physician_id,full_name,specialization,assigned_branch,physician_type,_rescued_data,_source_file,_ingest_timestamp,ingestion_date
DR001,Dr. Jose Co,Gastroenterology,H004,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR002,Dr. Kenneth Cruz,Internal Medicine,H007,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR003,Dr. Rafael Torres,Obstetrics,H010,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR004,Dr. Kenneth Villanueva,Endocrinology,H008,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR005,Dr. Miguel Sy,Infectious Disease,H006,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR006,Dr. Ramon Bautista,Nephrology,H007,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR007,Dr. Fe Gutierrez,Cardiology,H008,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR008,Dr. Cristina Delos Santos,Gastroenterology,H010,Consultant,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR009,Dr. Veronica Mendoza,Nephrology,H001,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05
DR010,Dr. Cristina Chan,Endocrinology,H002,Attending,null,abfss://data@datastoragea.dfs.core.windows.net/staging/physicians_raw/physicians_raw,2026-05-05T11:36:38.468Z,2026-05-05


In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

In [0]:
bronze_table = 'catalog.bronze.physicians_raw'
silver_table = 'catalog.silver.dim_physicians'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_physicians/checkpoint/"

df_physicians_bronze = spark.readStream.table(bronze_table)

df_physicians_clean = (
    df_physicians_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumnRenamed("full_name", "physician_name")
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_physicians(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["physician_id"])
    
    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_physicians = DeltaTable.forName(spark, silver_table)

    (dim_physicians.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.physician_id = s.physician_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_physicians_clean.writeStream
        .foreachBatch(merge_dim_physicians)  # for every batch, merge_dim_physicians is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_physicians LIMIT 10;

physician_id,physician_name,specialization,assigned_branch,physician_type,load_timestamp
DR015,Dr. Rafael Macaraeg,Internal Medicine,H006,Attending,2026-05-23T16:20:46.841Z
DR022,Dr. Lorena Flores,Infectious Disease,H003,Attending,2026-05-23T16:20:46.841Z
DR004,Dr. Kenneth Villanueva,Endocrinology,H008,Attending,2026-05-23T16:20:46.841Z
DR005,Dr. Miguel Sy,Infectious Disease,H006,Attending,2026-05-23T16:20:46.841Z
DR041,Dr. Marco Macaraeg,Endocrinology,H003,Attending,2026-05-23T16:20:46.841Z
DR006,Dr. Ramon Bautista,Nephrology,H007,Attending,2026-05-23T16:20:46.841Z
DR023,Dr. Patricia Uy,Pulmonology,H009,Consultant,2026-05-23T16:20:46.841Z
DR046,Dr. Juan Gonzales,Nephrology,H003,Attending,2026-05-23T16:20:46.841Z
DR026,Dr. Maria Bautista,Oncology,H005,Attending,2026-05-23T16:20:46.841Z
DR025,Dr. Kenneth Uy,Obstetrics,H001,Attending,2026-05-23T16:20:46.841Z


In [0]:
# Data quality check
from pyspark.sql.functions import col, count

print("DIM PHYSICIANS")
df = spark.read.table("catalog.silver.dim_physicians")
total = df.count()
print(f"Total rows: {total} (expected: 50)")

dup_id = total - df.select("physician_id").distinct().count()
print(f"Duplicate physician_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["physician_id", "physician_name", "specialization",
          "assigned_branch", "physician_type"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

print("Specialization distribution:")
df.groupBy("specialization").count().orderBy("count", ascending=False).show()

print("Physician type distribution:")
df.groupBy("physician_type").count().show()

# Assigned branch should all be valid: H001-H010
invalid_branch = df.filter(
    ~col("assigned_branch").isin([f"H{i:03d}" for i in range(1,11)])
).count()
print(f"Invalid assigned_branch: {'OK' if invalid_branch == 0 else f'CHECK — {invalid_branch} rows'}")

DIM PHYSICIANS
Total rows: 50 (expected: 50)
Duplicate physician_ids: OK
  OK physician_id: 0 nulls
  OK physician_name: 0 nulls
  OK specialization: 0 nulls
  OK assigned_branch: 0 nulls
  OK physician_type: 0 nulls
Specialization distribution:
+------------------+-----+
|    specialization|count|
+------------------+-----+
|        Obstetrics|    7|
|Infectious Disease|    7|
|         Neurology|    5|
|        Cardiology|    5|
|        Nephrology|    5|
|       Pulmonology|    5|
|  Gastroenterology|    5|
|     Endocrinology|    5|
|          Oncology|    4|
| Internal Medicine|    2|
+------------------+-----+

Physician type distribution:
+--------------+-----+
|physician_type|count|
+--------------+-----+
|     Attending|   40|
|    Consultant|   10|
+--------------+-----+

Invalid assigned_branch: OK
